In [1]:
%%bash

wget -O "./protein/Apis_mellifera.faa.gz" "https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/pep/Apis_mellifera.Amel_HAv3.1.pep.all.fa.gz"
gunzip "./protein/Apis_mellifera.faa.gz"

--2026-11-01 13:27:22--  https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/pep/Apis_mellifera.Amel_HAv3.1.pep.all.fa.gz
Resolving ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)... 193.62.193.161
Connecting to ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)|193.62.193.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5385788 (5.1M) [application/x-gzip]
Saving to: ‘./protein/Apis_mellifera.faa.gz’

     0K .......... .......... .......... .......... ..........  0%  240K 22s
    50K .......... .......... .......... .......... ..........  1%  152K 28s
   100K .......... .......... .......... .......... ..........  2% 18.2M 18s
   150K .......... .......... .......... .......... ..........  3% 1.76M 14s
   200K .......... .......... .......... .......... ..........  4%  326K 14s
   250K .......... .......... .......... .......... ..........  5%  106M 12s
   300K .......... .......... .......... .......... .....

确定 adata 的 gene_id 与蛋白组中的 header 都是 GTF 中特定列的子集：

In [2]:
import scanpy as sc
import pandas as pd
from pyfaidx import Fasta

/home/liuzy/Software/miniconda3/envs/saturn/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [3]:
Amel_gtf = pd.read_csv("./metadata/Apis_mellifera.tsv", sep="\t")
#! Apis_mellifera.tsv: see https://github.com/zliu90693/anno_cells/tree/main/.subprojects/MetaNeighbor/metadata/Apis_mellifera.tsv

In [4]:
Amel_gtf_gid = set(Amel_gtf["gene_id"])
Amel_gtf_pid = set(Amel_gtf["protein_id"])

In [ ]:
Amel_adata = sc.read_h5ad("./matrix/Amel_wll.h5ad")
#! Amel_wll.h5ad，see https://github.com/zliu90693/anno_cells/blob/main/.subprojects/MetaNeighbor/01_01-get_n_check_data-Amel.ipynb

In [9]:
Amel_adata

AnnData object with n_obs × n_vars = 115169 × 12292
    obs: 'sample', 'amel_key', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'f', 'n', 'q', 'integrated_snn_res.1', 'seurat_clusters', 'type', 'cell_type'

In [10]:
Amel_adata_gid = set(Amel_adata.var.index)

In [ ]:
Amel_protein = Fasta("./protein/Apis_mellifera.faa")
Amel_protein_pid = set(header for header in Amel_protein.keys())

In [12]:
print(len(Amel_gtf_gid), len(Amel_adata_gid), len(Amel_gtf_gid & Amel_adata_gid))

12398 12292 12292


In [19]:
print(len(Amel_gtf_pid), len(Amel_protein_pid), len(Amel_gtf_pid & Amel_protein_pid))

23471 23470 23470


构建 protein-gene 对应关系

In [21]:
import json
import pandas as pd

Amel_gtf = pd.read_csv("metadata/Apis_mellifera.tsv", sep="\t", low_memory=False)

m = Amel_gtf.dropna(subset=["gene_id", "protein_id"])[["gene_id", "protein_id"]]
m = m.drop_duplicates()
m["gene_id"] = m["gene_id"].astype(str)
m["protein_id"] = m["protein_id"].astype(str)

gene_to_proteins = {}
for gene, sub in m.groupby("gene_id"):
    gene_to_proteins[gene] = sorted(set(sub["protein_id"]))

with open("./gene_protein_map/Amel.gene_id_to_protein_ID.json", "w") as f:
    json.dump(gene_to_proteins, f, indent=4, sort_keys=True)